# LG CNS 사내 공유: LangGraph 기반 AI Agent 고도화 실습

> 목표: 기존 Colab/LG CNS 실습을 기반으로, **회사 내부 공유에서 설명 가능한 수준의 LangGraph 멀티 에이전트 아키텍처**를 구성하고 검증한다.

이 노트북은 단순히 코드가 돌아가는 예제가 아니라, 회사 내부 공유에서 다음 흐름을 함께 설명하고 검증할 수 있도록 만든다.

- LangGraph의 핵심 문법: `State`, `Node`, `Edge`, `START`, `END`, `Reducer`
- 로컬 우선 정보 수집 MAS: `DB Agent + CSV Agent + Web Agent + RAG Agent`
- 정보가 없을 때 RAG Agent로 fallback하는 설계
- 그래프/X-Ray 시각화로 구조 검증
- 운영 고도화 포인트: Hybrid Search, RRF, Tool Observability, Golden Test, LLM-as-Judge, LangSmith


## 0. 목차

이번 공유는 처음부터 복잡한 멀티 에이전트로 들어가지 않는다. **작은 LLM 호출에서 시작해 LangGraph 멀티 에이전트까지 단계적으로 확장**한다.

| 시간 | 주제 | 목표 |
|---:|---|---|
| 0~15분 | 핵심 개념과 Lang 생태계 | Agent, Tool, Skill, Workflow, LangChain/LangGraph/LangSmith 역할을 맞춘다 |
| 15~35분 | LLM 직접 호출과 LCEL | `invoke`, Prompt, Parser, Chain 구조를 이해한다 |
| 35~55분 | Prompt / Parser / Memory | 출력 형식 제어와 대화 기억의 필요성을 이해한다 |
| 55~75분 | LangGraph 기본 그래프 | `StateGraph`, `State`, `Node`, `Edge`를 코드로 익힌다 |
| 75~95분 | 조건부/병렬 그래프 | 라우팅, fan-out, reporter 합류 구조를 만든다 |
| 95~115분 | 로컬 우선 MAS + RAG fallback | DB/CSV/Web/RAG Agent를 조합한다 |
| 115~120분 | 운영 고도화 정리 | LangSmith, 평가, 관측성, RRF를 운영 관점으로 연결한다 |

전체 난이도 계단은 다음과 같다.

```mermaid
flowchart LR
    A[LLM 직접 호출] --> B[LCEL Chain]
    B --> C[Prompt / Parser]
    C --> D[Memory]
    D --> E[LangGraph 직렬]
    E --> F[조건부 / 병렬]
    F --> G[MAS + RAG fallback]
    G --> H[테스트 / 관측 / 평가]
```

공유 포인트:

- `LLM`은 답변 생성기, `Agent`는 목표를 보고 도구를 활용하는 실행 구조다.
- `Workflow`는 사람이 정한 흐름이고, `Agent`는 LLM이 판단하는 영역이다.
- 회사 업무에서는 **정해진 흐름은 LangGraph로 통제**하고, **애매한 검색/판단은 Agent 또는 RAG에 위임**하는 구조가 안정적이다.


## 1. 먼저 알고 가야 할 핵심 개념

이 강의에서 코드를 이해하려면 아래 개념을 먼저 잡고 가면 된다. 이름이 비슷해서 헷갈리지만, 역할이 다르다.

### 1-1. AI Agent

AI Agent는 LLM을 단순 답변 생성기가 아니라 **판단하고 행동하는 실행 주체**로 쓰는 구조다.

일반 LLM 호출은 보통 이렇게 끝난다.

```mermaid
flowchart LR
    User[사용자 질문] --> LLM[LLM]
    LLM --> Answer[답변]
```

Agent는 중간에 판단과 행동이 들어간다.

```mermaid
flowchart LR
    User[사용자 요청] --> Agent[Agent]
    Agent --> Think[판단 / 계획]
    Think --> Tool[도구 선택]
    Tool --> Observe[결과 관찰]
    Observe --> Agent
    Agent --> Answer[최종 답변]
```

핵심:

- Agent는 “무조건 답변만 하는 LLM”이 아니다.
- Agent는 목표를 보고 어떤 도구를 쓸지 판단할 수 있다.
- 단, 판단을 LLM에게 맡기므로 항상 100% 신뢰하면 안 된다.
- 회사 업무에서는 중요한 흐름은 Graph로 고정하고, 애매한 판단만 Agent에게 맡기는 편이 안정적이다.

### 1-2. Tool

Tool은 Agent가 사용할 수 있는 **외부 기능**이다.

예:

- DB 조회 도구
- CSV 파일 읽기 도구
- 웹 검색 도구
- 계산기 도구
- 사내 문서 검색 도구
- 이메일/슬랙 발송 도구

LangChain에서는 보통 함수에 `@tool`을 붙여 Tool로 만든다.

```python
@tool
def search_news(keyword: str):
    """뉴스를 검색할 때 사용한다."""
    ...
```

중요한 점:

- Tool은 그냥 Python 함수일 수 있다.
- `@tool`을 붙이면 함수 이름, 인자, docstring이 LLM에게 전달된다.
- LLM은 docstring을 보고 “이 도구를 언제 써야 하는지” 판단한다.
- 도구를 실제로 실행하는 주체는 LangGraph의 `ToolNode`나 Agent runtime이다.

### 1-3. Skill

Skill은 Tool보다 더 넓은 개념이다.

Tool이 “실행 가능한 함수 하나”에 가깝다면, Skill은 **특정 작업을 잘 수행하기 위한 지식, 절차, 규칙, 도구 묶음**에 가깝다.

예:

- “PDF에서 표를 추출하는 Skill”
- “LangGraph 워크플로우를 설계하는 Skill”
- “회사 보고서 형식으로 요약하는 Skill”
- “RAG 인덱스를 만들고 검색 품질을 평가하는 Skill”

비교하면 다음과 같다.

| 구분 | 의미 | 예시 |
|---|---|---|
| Tool | Agent가 호출할 수 있는 함수 | `news_search_tool()`, `db_query_tool()` |
| Skill | 작업을 잘하기 위한 절차/지식/도구 묶음 | OCR 처리 절차, RAG 구축 절차, 보고서 작성 방식 |
| Agent | 목표를 보고 Tool/Skill을 활용하는 실행 주체 | 뉴스 수집 Agent, 리포트 작성 Agent |

### 1-4. Workflow

Workflow는 사람이 미리 정해둔 작업 순서다.

```mermaid
flowchart LR
    A[입력] --> B[전처리]
    B --> C[조회]
    C --> D[리포트 생성]
    D --> E[출력]
```

특징:

- 순서가 명확하다.
- 테스트하기 쉽다.
- 운영 안정성이 높다.
- 입력이 달라져도 큰 흐름은 유지된다.

반대로 Agent는 입력에 따라 도구 선택이나 실행 순서가 달라질 수 있다.

### 1-5. LangGraph

LangGraph는 LLM/Agent/Workflow를 **그래프 형태로 설계하고 실행하는 프레임워크**다.

```mermaid
flowchart TD
    START([START]) --> Node1[Node]
    Node1 --> Router{조건 판단}
    Router -->|A| NodeA[Node A]
    Router -->|B| NodeB[Node B]
    NodeA --> END([END])
    NodeB --> END
```

LangGraph를 쓰는 이유:

- Agent 흐름을 눈으로 볼 수 있다.
- 분기, 반복, 병렬 실행을 명시적으로 만들 수 있다.
- 각 단계의 결과를 `State`에 남길 수 있다.
- 중간에 사람이 개입하는 Human-in-the-loop를 붙이기 쉽다.
- Checkpointer를 붙여 세션별 대화를 이어갈 수 있다.

### 1-6. State, Node, Edge

LangGraph 코드는 거의 이 세 단어로 읽으면 된다.

| 개념 | 쉬운 설명 | 코드에서 보이는 형태 |
|---|---|---|
| State | 그래프 전체가 공유하는 데이터 가방 | `class AgentState(TypedDict)` |
| Node | State를 받아 작업하고 일부 State를 반환하는 함수 | `def db_agent(state): ...` |
| Edge | 노드와 노드 사이의 이동 경로 | `builder.add_edge("A", "B")` |

흐름은 이렇게 이해하면 된다.

```mermaid
flowchart LR
    State1[현재 State] --> Node[Node 실행]
    Node --> Partial[일부 State 반환]
    Partial --> Merge[기존 State와 병합]
    Merge --> State2[다음 State]
```

### 1-7. Reducer

Reducer는 여러 노드가 같은 State key에 값을 쓸 때, 그 값을 어떻게 합칠지 정하는 규칙이다.

예를 들어 `db_agent`와 `csv_agent`가 동시에 `evidence`를 반환하면 둘 중 하나가 사라지면 안 된다.

```python
evidence: Annotated[list[str], operator.add]
```

이 뜻은 다음과 같다.

- 기존 `evidence` 리스트에
- 새로 들어온 `evidence` 리스트를
- 덮어쓰지 말고 이어 붙여라.

`messages`에는 LangGraph에서 제공하는 `add_messages` reducer를 자주 쓴다.

```python
messages: Annotated[Sequence[BaseMessage], add_messages]
```

### 1-8. RAG Agent

RAG Agent는 내부 문서나 벡터 DB에서 관련 근거를 찾아 답변에 붙여주는 Agent다.

DB/CSV/Web과 RAG의 차이는 다음과 같다.

| 출처 | 강점 | 약점 |
|---|---|---|
| DB | 정확한 key 조회 | 등록되지 않은 표현에 약함 |
| CSV | 만들기 쉽고 실습에 좋음 | 규모가 커지면 관리 어려움 |
| Web | 최신 정보 | 신뢰도와 보안 이슈 |
| RAG | 내부 문서 의미 검색 | 문서 품질, 청킹, 임베딩 품질에 영향 |

이번 노트북에서는 `고혈압`처럼 정확한 키가 있으면 DB/CSV를 먼저 쓰고, `혈압 높은 사람`처럼 표현이 다르면 RAG로 보강한다.

### 1-9. Checkpointer와 Store

LangGraph에서 “기억”은 크게 두 종류로 나눠 이해하면 쉽다.

| 구분 | 저장 대상 | 범위 | 예시 |
|---|---|---|---|
| Checkpointer | 그래프 실행 중 State | 같은 `thread_id` 안 | 대화 이어가기, interrupt 재개 |
| Store | 여러 세션에서 재사용할 지식 | thread를 넘어 공유 | 사용자 선호, 장기 기억, 사내 지식 |

실습에서는 `InMemorySaver`, `SqliteSaver`, `InMemoryStore` 같은 개념이 나온다.

- `InMemorySaver`: 메모리에 저장. 런타임 종료 시 사라짐.
- `SqliteSaver`: SQLite 파일에 저장. 런타임이 끝나도 유지됨.
- 운영에서는 보통 PostgreSQL, MySQL, Redis, Vector DB 같은 저장소를 붙인다.

### 1-10. LangSmith

LangSmith는 LLM/Agent 실행을 추적하고 평가하는 도구다.

확인할 수 있는 것:

- 어떤 노드가 실행됐는지
- LLM 호출이 몇 번 발생했는지
- 토큰과 비용이 얼마나 들었는지
- 어느 도구에서 실패했는지
- 응답 시간이 어디서 길어졌는지

회사에서 Agent를 운영하려면 “답변이 잘 나왔다”보다 “왜 그렇게 나왔는지 추적 가능하다”가 더 중요하다.

## 1-1. Lang 생태계 아키텍처

LangGraph를 바로 코드로 보면 `State`, `Node`, `Edge` 같은 문법이 먼저 보인다. 하지만 실제로는 LangChain 생태계 안에서 각 도구가 역할을 나눠 가진다.

```mermaid
flowchart TB
    User[사용자 / 업무 요청] --> App[우리 애플리케이션 코드]

    App --> LC[LangChain
Prompt, Model, Parser, Tool, LCEL]
    App --> LG[LangGraph
State, Node, Edge, Workflow, Agent Graph]

    LC --> Provider[Model Provider
OpenAI, Anthropic, Local LLM]
    LC --> Tools[External Tools
DB, API, Search, Files]

    LG --> LC
    LG --> Checkpoint[Checkpointer / Store
Memory, SQLite, Postgres]

    App --> LS[LangSmith
Trace, Token, Latency, Evaluation]
    LG --> LS
    LC --> LS
```

각 구성요소를 한 줄로 정리하면 다음과 같다.

| 구성요소 | 역할 | 이 강의에서의 위치 |
|---|---|---|
| LangChain | LLM 호출, Prompt, Parser, Tool, LCEL 체인을 다루는 기본 도구 모음 | `Prompt \| LLM \| Parser`, `@tool`, `ChatOpenAI` |
| LCEL | LangChain 안에서 여러 실행 단계를 `\|`로 연결하는 표현 문법 | `chain = prompt \| llm \| parser` |
| LangGraph | 여러 노드와 조건부 흐름을 State 중심으로 연결하는 그래프 프레임워크 | `StateGraph`, `add_node`, `add_edge`, `compile` |
| LangSmith | 실행 과정을 추적하고 평가하는 관측/테스트 도구 | trace, token, latency, LLM-as-Judge |
| Model Provider | 실제 LLM을 제공하는 외부/로컬 모델 서비스 | `ChatOpenAI(model="gpt-4o-mini")` |
| Tool | LLM 또는 Agent가 호출할 수 있는 함수/외부 기능 | DB 조회, CSV 조회, 기상청 RSS, RAG 검색 |
| Checkpointer / Store | 그래프 상태와 장기 기억을 저장하는 계층 | Human-in-the-loop, 세션 재개, 운영 메모리 |

중요한 구분:

- **LangChain**은 LLM을 호출하고 결과를 다루는 부품 상자에 가깝다.
- **LangGraph**는 그 부품들을 어떤 순서와 조건으로 실행할지 정하는 업무 흐름 엔진에 가깝다.
- **LangSmith**는 그 흐름이 실제로 어떻게 돌았는지 관찰하고 평가하는 모니터링/평가 도구에 가깝다.

이 강의에서는 먼저 LangChain의 기본 부품을 보고, 그다음 LangGraph로 업무 흐름을 고정하고, 마지막에 LangSmith/평가/휴먼인더루프로 운영 관점까지 확장한다.


## 2. API Key 설정

OpenAI API 키는 코드에 직접 적지 않는다.

- 이미 터미널/VS Code/Jupyter 환경에 `OPENAI_API_KEY`가 있으면 그대로 읽는다.
- 없으면 실행 중에 `getpass()`로 입력한다.
- 노트북 저장 시 키 값이 남지 않도록 주의한다.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY를 입력하세요: ")

print("OPENAI_API_KEY:", "설정됨" if os.environ.get("OPENAI_API_KEY") else "없음")

## 2-1. 선택: LangSmith Trace 설정

LangSmith는 LLM, Tool, LangGraph 실행을 추적해서 다음을 확인할 수 있게 해준다.

- 어떤 그래프 노드가 실행됐는가?
- 어떤 LLM 호출이 발생했는가?
- 어떤 Tool이 어떤 입력으로 호출됐는가?
- 토큰 수와 응답 시간은 얼마나 나왔는가?
- 에이전트가 어느 단계에서 실패했는가?

아래 셀은 `LANGSMITH_API_KEY`가 있으면 자동으로 tracing을 켠다. 키가 없으면 실행 중에 입력받는다.

> 공유용 노트북에는 LangSmith 키를 직접 저장하지 않는다.


In [ ]:
import os
from getpass import getpass

os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = os.environ.get("LANGSMITH_PROJECT", "ai-agent-class")

if not os.environ.get("LANGSMITH_API_KEY"):
    langsmith_key = getpass("LANGSMITH_API_KEY를 입력하세요. 건너뛰려면 Enter: ")
    if langsmith_key.strip():
        os.environ["LANGSMITH_API_KEY"] = langsmith_key.strip()

if os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = "true"
    print("LangSmith tracing enabled:", os.environ["LANGSMITH_PROJECT"])
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing disabled: API key가 없습니다.")


### 2-2. LangSmith Trace 확인용 LangGraph 미니 예제

아래 셀은 LangSmith가 실제로 LangGraph 실행을 잡는지 확인하는 최소 예제다.

실행 후 LangSmith 화면에서 `Tracing -> ai-agent-class` 프로젝트를 보면 다음 흐름이 보여야 한다.

```mermaid
flowchart LR
    START --> answer_node
    answer_node --> END
```

확인 포인트:

- `graph.invoke(...)` 호출 1건이 trace로 남는가?
- 내부의 `answer_node` 실행이 보이는가?
- `ChatOpenAI` 호출의 latency/token 정보가 보이는가?


In [ ]:
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

class TraceState(TypedDict):
    question: str
    answer: str

trace_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def answer_node(state: TraceState):
    # 이 LLM 호출까지 LangSmith trace에서 확인할 수 있다.
    response = trace_llm.invoke(
        "다음 질문에 한 문장으로 답해줘: " + state["question"]
    )
    return {"answer": response.content}

trace_builder = StateGraph(TraceState)
trace_builder.add_node("answer_node", answer_node)
trace_builder.add_edge(START, "answer_node")
trace_builder.add_edge("answer_node", END)
trace_graph = trace_builder.compile()

trace_result = trace_graph.invoke({"question": "LangSmith trace가 무엇을 보여주나요?"})
print(trace_result["answer"])
print("LangSmith project:", os.environ.get("LANGSMITH_PROJECT"))


## 3. 기초 단계 실습: LLM에서 Agent로 가는 길

LangGraph 멀티 에이전트를 보기 전에, 먼저 LLM 호출이 어떻게 체인과 에이전트로 확장되는지 확인한다.

```mermaid
flowchart LR
    A[LLM 직접 호출] --> B[LangChain]
    B --> C[LCEL: Prompt → LLM → Parser]
    C --> D[Prompt 최적화]
    D --> E[Parser로 출력 제어]
    E --> F[Memory로 대화 유지]
    F --> G[Agent / LangGraph]
```

여기서는 뒤의 LangGraph 실습을 이해하기 위해 필요한 최소 코드만 단계별로 본다.


### 3-1. LLM 직접 호출: `invoke()`부터 이해하기

`invoke()`는 “실제로 실행한다”는 뜻이다.

- `llm.invoke(prompt)`는 LLM에게 프롬프트를 보내고 응답을 받는다.
- LangGraph의 `graph.invoke(input)`도 같은 감각이다. 그래프 전체를 실행한다.
- LCEL의 `chain.invoke(input)`도 체인을 실행한다.

즉 이 강의에서 `invoke`는 대부분 **정의한 것을 실제로 한 번 실행한다**로 읽으면 된다.

In [ ]:
from langchain_openai import ChatOpenAI

# ChatOpenAI는 OpenAI Chat 모델을 LangChain 방식으로 쓰기 위한 객체다.
# temperature=0은 같은 입력에 대해 최대한 일관된 답변을 받기 위한 설정이다.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# invoke는 LLM에게 실제 요청을 보내고 응답 객체를 받는다.
response = llm.invoke("AI Agent를 한 문장으로 설명해줘")

# response.content에는 LLM이 생성한 최종 텍스트가 들어 있다.
print(response.content)

### 3-2. LCEL 기본형: `Prompt | LLM | Parser`

여기서 처음 나오는 `LCEL`은 **LangChain Expression Language**의 줄임말이다.

쉽게 말하면, **LLM 작업 단계를 `|` 기호로 이어서 하나의 실행 가능한 체인으로 만드는 문법**이다.

예를 들어 아래 코드는:

```python
chain = prompt | llm | parser
```

다음 흐름으로 읽으면 된다.

```mermaid
flowchart LR
    Input[사용자 입력] --> Prompt[Prompt
질문 양식 만들기]
    Prompt --> LLM[LLM
답변 생성]
    LLM --> Parser[Parser
출력 정리]
    Parser --> Output[최종 결과]
```

각 단계의 역할은 다음과 같다.

- Prompt: 사용자 입력을 LLM에게 보낼 질문 형태로 만든다.
- LLM: 실제 모델을 호출해서 답변을 생성한다.
- Parser: LLM 응답을 문자열, JSON, 리스트처럼 원하는 형태로 정리한다.
- `|`: 앞 단계의 출력을 다음 단계 입력으로 넘긴다.

즉, 이 강의에서 `LCEL 체인`은 **Prompt → LLM → Parser를 레고처럼 연결한 실행 파이프라인**으로 이해하면 된다.


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# PromptTemplate은 변수가 들어갈 수 있는 질문 양식이다.
prompt = PromptTemplate.from_template(
    "{topic}를 LangChain/LangGraph 프레임워크 문맥에서 처음 배우는 사람에게 한국어로 3문장 이내로 설명해줘. "
    "일반적인 그래프 이론이나 NLP 모델이 아니라, AI Agent 워크플로우 프레임워크로 설명해줘."
)

# StrOutputParser는 LLM 응답 객체에서 문자열만 깔끔하게 꺼내준다.
parser = StrOutputParser()

# LCEL 체인: prompt 결과를 llm에 넣고, llm 결과를 parser에 넣는다.
chain = prompt | llm | parser

# invoke에 딕셔너리를 넣으면 {topic} 자리에 값이 주입된다.
result = chain.invoke({"topic": "LangGraph"})
print(result)

### 3-3. ChatPromptTemplate: 역할을 나눠 프롬프트 만들기

단순 문자열 프롬프트보다 실무에서는 `system`, `human`, `ai` 역할을 나누는 방식이 더 안정적이다.

- `system`: LLM의 역할, 규칙, 말투를 정한다.
- `human`: 사용자의 입력이다.
- `ai`: 이전 AI 응답 예시나 대화 흐름을 넣을 수 있다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# ChatPromptTemplate은 system/human 같은 메시지 역할을 분리해서 프롬프트를 만든다.
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 LG CNS 사내 교육을 돕는 AI Agent 강사다. Tool은 LLM이 호출할 수 있는 함수/외부 기능, Agent는 목표 달성을 위해 LLM이 판단하고 Tool을 사용할 수 있는 실행 주체로 설명한다. 설명은 짧고 명확하게 한다."),
    ("human", "{question}"),
])

# 역할 기반 프롬프트도 똑같이 LCEL로 연결할 수 있다.
chat_chain = chat_prompt | llm | StrOutputParser()

# question 변수에 사용자 질문이 들어간다.
answer = chat_chain.invoke({"question": "Tool과 Agent의 차이를 알려줘"})
print(answer)

### 3-4. Parser: LLM 출력을 구조화하기

LLM은 가끔 “긍정입니다”, “아마 긍정 같아요”처럼 제멋대로 답한다. 실무에서는 후속 코드가 읽기 쉬운 형식으로 출력하게 만들어야 한다.

Parser는 LLM 출력을 문자열, JSON, 리스트, Enum 같은 구조로 바꾸는 역할을 한다.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

# JsonOutputParser는 LLM 출력이 JSON으로 나오도록 유도하고 파싱한다.
json_parser = JsonOutputParser()

# format_instructions에는 JSON으로 답하라는 안내 문구가 들어 있다.
format_instructions = json_parser.get_format_instructions()

# 프롬프트에 출력 형식을 명시적으로 넣는다.
json_prompt = PromptTemplate.from_template(
    """
다음 문장을 감정 분석해줘.
반드시 JSON 형식으로만 답해.

문장: {review}
출력 형식 안내: {format_instructions}
""".strip()
)

# Prompt -> LLM -> JSON Parser 순서로 실행한다.
json_chain = json_prompt | llm | json_parser

# 결과는 문자열이 아니라 Python dict에 가깝게 다룰 수 있다.
parsed = json_chain.invoke({
    "review": "배송도 빠르고 품질도 좋아서 만족합니다.",
    "format_instructions": format_instructions,
})

print(parsed)
print(type(parsed))

### 3-5. RunnableParallel: 한 질문을 여러 관점으로 동시에 처리하기

`RunnableParallel`은 하나의 입력을 여러 체인에 동시에 넣고 결과를 딕셔너리로 모으는 방식이다.

LangGraph 병렬 fan-out을 배우기 전에, LCEL 병렬 처리를 먼저 보면 이해가 쉽다.

In [ ]:
from langchain_core.runnables import RunnableParallel

# 같은 입력을 요약 체인과 키워드 체인에 동시에 보낼 것이다.
summary_chain = PromptTemplate.from_template(
    "다음 내용을 한 문장으로 요약해줘: {text}"
) | llm | StrOutputParser()

keyword_chain = PromptTemplate.from_template(
    "다음 내용의 핵심 키워드 3개만 쉼표로 출력해줘: {text}"
) | llm | StrOutputParser()

# RunnableParallel은 여러 체인을 동시에 실행하고 결과를 dict로 모은다.
parallel_chain = RunnableParallel({
    "summary": summary_chain,
    "keywords": keyword_chain,
})

parallel_result = parallel_chain.invoke({
    "text": "LangGraph는 State를 중심으로 노드와 엣지를 연결해 에이전트 워크플로우를 구성한다."
})

print(parallel_result)

### 3-6. Memory: 대화 내용을 기억하게 하기

기본 LLM 호출은 이전 대화를 자동으로 기억하지 않는다. 기억이 필요하면 히스토리를 별도로 저장하고 다시 프롬프트에 넣어야 한다.

LangChain에서는 `RunnableWithMessageHistory`를 이용해 체인에 세션별 대화 기록을 붙일 수 있다.

LangGraph에서는 뒤에서 `messages: Annotated[..., add_messages]`와 checkpointer가 비슷한 역할을 한다.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 세션별 대화 기록을 저장할 간단한 메모리 저장소다.
# 운영에서는 Redis, DB, LangGraph Checkpointer 등으로 대체할 수 있다.
session_histories = {}


def get_session_history(session_id: str):
    """session_id별로 독립된 대화 히스토리를 꺼낸다."""
    if session_id not in session_histories:
        session_histories[session_id] = InMemoryChatMessageHistory()
    return session_histories[session_id]


# MessagesPlaceholder는 과거 대화 메시지가 들어갈 자리다.
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "너는 사용자의 이름을 기억하는 친절한 비서다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

# 일반 체인 자체는 기억이 없다.
memory_chain = memory_prompt | llm | StrOutputParser()

# RunnableWithMessageHistory가 체인 주변에 세션별 기억 기능을 붙인다.
chain_with_memory = RunnableWithMessageHistory(
    memory_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

# 같은 session_id를 쓰면 이전 대화가 이어진다.
config = {"configurable": {"session_id": "company-demo-user"}}

print(chain_with_memory.invoke({"question": "내 이름은 승찬이야."}, config=config))
print(chain_with_memory.invoke({"question": "내 이름이 뭐라고 했지?"}, config=config))

### 3-7. 여기까지의 연결

지금까지는 LangGraph 없이도 가능한 기초다.

- `llm.invoke()`로 LLM을 직접 호출했다.
- `Prompt → LLM → Parser`로 LCEL 체인을 만들었다.
- Prompt를 역할 기반으로 만들었다.
- Parser로 출력 형식을 제어했다.
- RunnableParallel로 여러 체인을 동시에 실행했다.
- Memory로 대화 기록을 이어갔다.

이제부터는 여러 체인과 노드를 연결하고, 조건부 분기와 병렬 실행을 명확하게 관리하기 위해 LangGraph를 쓴다.

## 4. LangGraph 기본 개념

LangGraph는 LLM 애플리케이션을 **상태 기반 그래프**로 표현한다.

```mermaid
flowchart TD
    START([START]) --> NodeA[Node A]
    NodeA --> NodeB[Node B]
    NodeB --> END([END])
```

핵심 구성요소:

- `State`: 노드들이 공유하는 데이터 구조
- `Node`: `state`를 받아 일부 state를 반환하는 함수
- `Edge`: 노드 간 실행 순서
- `Reducer`: 같은 key에 여러 노드가 동시에 값을 쓸 때 합치는 규칙
- `compile()`: 설계를 실제 실행 가능한 그래프로 변환
- `invoke()`: 그래프를 실제로 실행

## 4-0. `TypedDict`와 State 내부 구조 시각화

LangGraph에서 `State`는 노드들이 함께 읽고 쓰는 **공유 데이터 상자**다. Python에서는 보통 `TypedDict`로 State의 모양을 정의한다.

예를 들어 아래 한 줄은 처음 보면 복잡해 보인다.

```python
messages: Annotated[Sequence[BaseMessage], add_messages]
```

이 한 줄은 3개로 나눠서 읽으면 된다.

```mermaid
flowchart LR
    A[messages] --> B[Sequence
여러 메시지가 순서대로 들어가는 목록]
    B --> C[BaseMessage
HumanMessage / AIMessage / ToolMessage의 부모 타입]
    A --> D[Annotated]
    D --> E[add_messages
새 메시지를 기존 messages 뒤에 이어 붙이는 reducer]
```

즉 의미는 다음과 같다.

| 코드 조각 | 뜻 |
|---|---|
| `messages` | State 안에 들어갈 key 이름 |
| `Sequence[BaseMessage]` | 값은 메시지 객체들이 순서대로 들어 있는 목록 구조 |
| `BaseMessage` | LangChain 메시지의 공통 부모 타입. `HumanMessage`, `AIMessage`, `ToolMessage` 등이 여기에 속한다 |
| `Annotated[..., add_messages]` | 이 key에 새 값이 들어오면 덮어쓰지 말고 `add_messages` 규칙으로 합치라는 뜻 |

실제 State는 이런 딕셔너리처럼 생겼다고 보면 된다.

```python
state = {
    "messages": [
        HumanMessage(content="고혈압 관리법 알려줘"),
        AIMessage(content="저염식과 유산소 운동이 중요합니다."),
    ],
    "question": "고혈압 관리법 알려줘",
    "disease": "고혈압",
    "found": True,
}
```

시각적으로 보면 다음과 같다.

```mermaid
classDiagram
    class AgentState {
        messages: Annotated[Sequence[BaseMessage], add_messages]
        question: str
        disease: str
        found: bool
        web_sufficient: bool
        evidence: Annotated[list[str], operator.add]
        used_agents: Annotated[list[str], operator.add]
        trace: Annotated[list[str], operator.add]
    }

    class messages {
        HumanMessage
        AIMessage
        ToolMessage
    }

    class reducers {
        add_messages
        operator.add
    }

    AgentState --> messages : contains
    AgentState --> reducers : merge rule
```

가장 중요한 차이는 **덮어쓰기와 누적**이다.

```mermaid
flowchart TB
    A[Node가 반환한 값] --> B{Reducer가 있는 key인가?}
    B -- 없음 --> C[기존 값을 새 값으로 덮어씀]
    B -- 있음 --> D[기존 값과 새 값을 합침]

    C --> E[question, disease, found]
    D --> F[messages, evidence, trace]
```

예를 들어 `question: str`은 보통 새 값이 오면 덮어쓴다. 반면 `messages`는 대화 흐름이 쌓여야 하므로 `add_messages`로 누적한다.


## 4-1. 초반에 알아야 할 LangGraph 메서드

아래 메서드들은 뒤 코드에서 계속 반복된다. 처음에는 영어 이름보다 **역할**로 읽으면 된다.

| 메서드 / 객체 | 어디서 나오는가 | 한 줄 의미 | 이 노트북에서의 역할 |
|---|---|---|---|
| `StateGraph(AgentState)` | LangGraph | 그래프 설계용 도화지를 만든다 | 전체 워크플로우의 뼈대를 만든다 |
| `builder.add_node("이름", 함수)` | LangGraph | 실행할 작업 단계를 그래프에 등록한다 | `db_agent`, `csv_agent`, `rag_agent`, `reporter`를 노드로 넣는다 |
| `builder.add_edge(A, B)` | LangGraph | A가 끝나면 B로 이동하게 연결한다 | `START -> entry_node`, `reporter -> END` 같은 고정 흐름을 만든다 |
| `builder.add_conditional_edges(...)` | LangGraph | state를 보고 다음 노드를 조건부로 고른다 | 로컬 데이터가 있으면 DB/CSV, 없으면 Web/RAG로 보낸다 |
| `builder.compile()` | LangGraph | 설계한 그래프를 실행 가능한 객체로 바꾼다 | `graph.invoke(...)`를 호출할 수 있게 만든다 |
| `graph.invoke(input)` | LangGraph | 그래프를 실제로 한 번 실행한다 | 사용자 질문을 넣고 최종 state를 받는다 |
| `graph.get_graph()` | LangGraph | 그래프 구조를 꺼낸다 | Mermaid/PNG로 구조를 검증한다 |
| `draw_mermaid_png()` | LangGraph/Graph 객체 | 그래프를 그림으로 렌더링한다 | 공유 중에 구조를 눈으로 보여준다 |
| `Annotated[..., add_messages]` | Python typing + LangGraph | 같은 key에 새 메시지가 오면 기존 메시지 뒤에 붙인다 | `messages` 대화 기록을 누적한다 |
| `Annotated[..., operator.add]` | Python typing + LangGraph | 여러 노드가 만든 리스트를 이어 붙인다 | `evidence`, `trace`, `used_agents`를 누적한다 |
| `ChatOpenAI(...)` | LangChain | OpenAI chat model 객체를 만든다 | Reporter 문장 다듬기나 Judge 평가에 사용한다 |
| `llm.invoke(prompt)` | LangChain | LLM에게 실제 요청을 보내고 응답을 받는다 | 최종 답변을 자연어로 다듬는다 |

코드로 읽으면 이렇게 대응된다.

```python
builder = StateGraph(AgentState)       # 1. 그래프 도화지 생성
builder.add_node("db_agent", db_agent) # 2. 실행할 작업 등록
builder.add_edge(START, "entry_node")  # 3. 시작 지점 연결

builder.add_conditional_edges(          # 4. 조건에 따라 다음 노드 선택
    "entry_node",
    route_after_entry,
    ["db_agent", "csv_agent", "web_agent"],
)

graph = builder.compile()               # 5. 실행 가능한 그래프로 변환
result = graph.invoke({"messages": [...]}) # 6. 실제 실행
```

정리하면, LangGraph에서는 함수가 함수를 직접 부르는 것이 아니라 **그래프가 노드 실행 순서를 관리**한다.

In [ ]:
from typing import Annotated, Sequence  # Annotated는 State key에 reducer 규칙을 붙일 때 사용
from typing_extensions import TypedDict  # TypedDict는 State의 key와 자료형을 명시하는 데 사용
import operator  # operator.add는 병렬 노드 결과 리스트를 이어 붙이는 reducer로 사용
import csv  # CSV 로컬 지식 파일을 읽기 위한 표준 라이브러리
import re
import sqlite3  # SQLite 로컬 DB를 조회하기 위한 표준 라이브러리
import time
from pathlib import Path
from functools import wraps

from IPython.display import Image, Markdown, display
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

ROOT = Path.cwd()  # 현재 노트북이 실행되는 작업 폴더
DB_PATH = ROOT / "disease.db"
CSV_PATH = ROOT / "disease_info.csv"


def ensure_demo_data():
    """Git에는 DB/CSV를 올리지 않고, 실습 실행 시 로컬 샘플 데이터를 자동 생성한다."""
    if not DB_PATH.exists():
        with sqlite3.connect(DB_PATH) as conn:
            conn.execute("""
            CREATE TABLE disease (
                name TEXT PRIMARY KEY,
                diet TEXT,
                exercise TEXT
            )
            """)
            conn.executemany(
                "INSERT INTO disease VALUES (?, ?, ?)",
                [
                    ("고혈압", "저염식, 채소·생선 위주, 칼륨 풍부한 바나나·시금치", "빠르게 걷기 하루 30분, 수영"),
                    ("당뇨", "정제 탄수화물 제한, 통곡물·잎채소, 저혈당지수 음식", "식후 가벼운 걷기, 근력운동 주 3회"),
                    ("비만", "고단백 저칼로리, 채소·닭가슴살, 가공식품 줄이기", "유산소+근력 병행, 주 5회 이상"),
                ],
            )

    if not CSV_PATH.exists():
        with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["name", "symptom", "caution"])
            writer.writeheader()
            writer.writerows([
                {"name": "고혈압", "symptom": "두통, 뒷목 뻣뻣함, 어지럼증", "caution": "나트륨 과다 섭취 주의, 정기 혈압 측정"},
                {"name": "당뇨", "symptom": "잦은 갈증, 빈뇨, 피로, 체중감소", "caution": "혈당 급변 주의, 공복 운동 자제"},
                {"name": "비만", "symptom": "피로, 관절 부담, 호흡 곤란", "caution": "급격한 단식 금지, 무리한 운동 주의"},
            ])


ensure_demo_data()

print("workspace:", ROOT)
print("disease.db exists:", DB_PATH.exists())
print("disease_info.csv exists:", CSV_PATH.exists())


### 4-2. LangGraph 첫 번째 예제: 직렬 그래프

가장 먼저 볼 구조는 직렬 그래프다.

```mermaid
flowchart LR
    START([START]) --> Retrieve[retrieve]
    Retrieve --> Generate[generate]
    Generate --> END([END])
```

이 예제의 목적은 “LangGraph는 함수가 함수를 직접 호출하는 것이 아니라, 그래프가 실행 순서를 관리한다”는 감각을 잡는 것이다.

In [ ]:
class SerialState(TypedDict):
    question: str  # 사용자 질문
    context: str   # retrieve 노드가 찾아온 근거
    answer: str    # generate 노드가 만든 최종 답변


def retrieve(state: SerialState):
    """질문을 보고 필요한 정보를 찾아오는 노드."""
    question = state["question"]  # 현재 State에서 사용자 질문을 꺼낸다.

    # 실무에서는 이 부분이 DB 조회, 문서 검색, API 호출, RAG 검색이 된다.
    if "감기" in question:
        context = "감기에는 수분 섭취, 휴식, 따뜻한 차가 도움이 된다."
    else:
        context = "등록된 건강관리 정보가 없습니다."

    # 노드는 전체 State를 다 반환하지 않아도 된다. 바뀐 key만 반환하면 LangGraph가 병합한다.
    return {"context": context}


def generate(state: SerialState):
    """retrieve가 만든 context를 바탕으로 답변을 만드는 노드."""
    answer = f"질문: {state['question']}\n근거: {state['context']}"
    return {"answer": answer}


serial_builder = StateGraph(SerialState)       # 1. State 구조를 기준으로 그래프 도화지를 만든다.
serial_builder.add_node("retrieve", retrieve)  # 2. retrieve 함수를 retrieve 노드로 등록한다.
serial_builder.add_node("generate", generate)  # 3. generate 함수를 generate 노드로 등록한다.

serial_builder.add_edge(START, "retrieve")     # 4. 그래프 시작 후 retrieve를 먼저 실행한다.
serial_builder.add_edge("retrieve", "generate")# 5. retrieve가 끝나면 generate로 이동한다.
serial_builder.add_edge("generate", END)       # 6. generate가 끝나면 그래프를 종료한다.

graph_serial = serial_builder.compile()         # 7. 설계한 그래프를 실행 가능한 객체로 만든다.

serial_result = graph_serial.invoke({           # 8. 그래프를 실제 실행한다.
    "question": "감기 걸렸을 때 어떻게 해야 해?",
    "context": "",                            # 초기값은 비워 둔다. retrieve가 채운다.
    "answer": "",                             # 초기값은 비워 둔다. generate가 채운다.
})

print(serial_result["answer"])

### 4-3. LangGraph 두 번째 예제: 조건부 분기

조건부 분기는 `if`문을 그래프 흐름으로 만든 것이다.

```mermaid
flowchart TD
    START([START]) --> Lookup[lookup]
    Lookup --> Check{정보 있음?}
    Check -->|있음| Answer[answer]
    Check -->|없음| Fallback[fallback]
    Answer --> END([END])
    Fallback --> END
```

`add_conditional_edges()`가 핵심이다.

In [ ]:
class RouteState(TypedDict):
    disease: str  # 사용자가 입력한 질병명
    info: str     # 로컬에서 찾은 정보
    answer: str   # 최종 답변


HEALTH_INFO = {
    "고혈압": "저염식, 채소와 생선 위주 식단, 가벼운 유산소 운동",
    "당뇨": "정제 탄수화물 제한, 통곡물, 식후 걷기",
}


def lookup(state: RouteState):
    """로컬 딕셔너리에서 질병 정보를 찾는 노드."""
    disease = state["disease"]                  # State에서 질병명을 꺼낸다.
    info = HEALTH_INFO.get(disease, "")         # 없으면 빈 문자열을 넣는다.
    return {"info": info}                       # 찾은 정보를 State에 저장한다.


def route_by_info(state: RouteState):
    """info가 있으면 정상 답변 노드로, 없으면 fallback 노드로 보낸다."""
    if state["info"]:                           # 빈 문자열이 아니면 정보가 있다는 뜻이다.
        return "answer"
    return "fallback"


def answer(state: RouteState):
    """로컬 정보가 있을 때 실행되는 정상 답변 노드."""
    return {"answer": f"{state['disease']} 관리법: {state['info']}"}


def fallback(state: RouteState):
    """로컬 정보가 없을 때 실행되는 보험 노드."""
    return {"answer": f"{state['disease']} 정보는 로컬 DB에 없어 전문가 상담 또는 추가 검색이 필요합니다."}


route_builder = StateGraph(RouteState)              # 1. 조건부 그래프 도화지 생성
route_builder.add_node("lookup", lookup)            # 2. 정보 조회 노드 등록
route_builder.add_node("answer", answer)            # 3. 정상 답변 노드 등록
route_builder.add_node("fallback", fallback)        # 4. fallback 노드 등록

route_builder.add_edge(START, "lookup")             # 5. 시작하면 lookup을 먼저 실행
route_builder.add_conditional_edges(                 # 6. lookup 결과를 보고 다음 경로 결정
    "lookup",                                       # lookup 노드가 끝난 뒤
    route_by_info,                                  # 이 라우터 함수로 판단해서
    {
        "answer": "answer",                       # 라우터가 answer를 반환하면 answer 노드로 이동
        "fallback": "fallback",                   # 라우터가 fallback을 반환하면 fallback 노드로 이동
    },
)
route_builder.add_edge("answer", END)               # 7. 정상 답변 후 종료
route_builder.add_edge("fallback", END)             # 8. fallback 답변 후 종료

graph_route = route_builder.compile()                # 9. 실행 가능한 그래프로 컴파일

print(graph_route.invoke({"disease": "고혈압", "info": "", "answer": ""})["answer"])
print(graph_route.invoke({"disease": "감기", "info": "", "answer": ""})["answer"])

### 4-4. LangGraph 세 번째 예제: 병렬 fan-out 후 합류

병렬 fan-out은 여러 노드를 동시에 실행하고 결과를 하나의 노드로 모으는 구조다.

```mermaid
flowchart TD
    START([START]) --> Food[food_node]
    START --> Exercise[exercise_node]
    Food --> Report[report_node]
    Exercise --> Report
    Report --> END([END])
```

여기서 중요한 것이 Reducer다. `food_node`와 `exercise_node`가 둘 다 `notes`에 값을 추가하므로 `operator.add`로 리스트를 이어 붙인다.

In [ ]:
class ParallelState(TypedDict):
    topic: str                                  # 사용자가 알고 싶은 주제
    notes: Annotated[list[str], operator.add]  # 여러 노드 결과를 이어 붙일 리스트
    report: str                                 # 최종 리포트


def food_node(state: ParallelState):
    """식단 관점의 정보를 만드는 노드."""
    topic = state["topic"]
    return {"notes": [f"식단 관점: {topic}에는 저염식과 채소 섭취가 중요합니다."]}


def exercise_node(state: ParallelState):
    """운동 관점의 정보를 만드는 노드."""
    topic = state["topic"]
    return {"notes": [f"운동 관점: {topic}에는 무리하지 않는 유산소 운동이 좋습니다."]}


def report_node_simple(state: ParallelState):
    """병렬 노드들이 만든 notes를 합쳐 최종 리포트를 만든다."""
    report = "\n".join(f"- {note}" for note in state["notes"])
    return {"report": report}


parallel_builder = StateGraph(ParallelState)                    # 1. 병렬 그래프 도화지 생성
parallel_builder.add_node("food_node", food_node)               # 2. 식단 노드 등록
parallel_builder.add_node("exercise_node", exercise_node)       # 3. 운동 노드 등록
parallel_builder.add_node("report_node", report_node_simple)    # 4. 합류 노드 등록

parallel_builder.add_edge(START, "food_node")                   # 5. 시작점에서 food_node로 분기
parallel_builder.add_edge(START, "exercise_node")               # 6. 시작점에서 exercise_node로도 분기
parallel_builder.add_edge("food_node", "report_node")          # 7. food_node 결과를 report_node로 보냄
parallel_builder.add_edge("exercise_node", "report_node")      # 8. exercise_node 결과도 report_node로 보냄
parallel_builder.add_edge("report_node", END)                   # 9. report_node가 끝나면 종료

graph_parallel = parallel_builder.compile()                      # 10. 실행 가능한 그래프로 컴파일

parallel_result = graph_parallel.invoke({                        # 11. 그래프 실행
    "topic": "고혈압 관리",
    "notes": [],
    "report": "",
})

print(parallel_result["report"])

### 4-5. 미니 예제에서 MAS로 확장하기

지금까지의 세 예제를 합치면 뒤의 멀티 에이전트 구조가 된다.

- 직렬 그래프: `조회 -> 답변 생성`
- 조건부 분기: `로컬 정보 있음? 없으면 fallback`
- 병렬 fan-out: `DB Agent와 CSV Agent를 동시에 실행 후 Reporter로 합류`

뒤에서 만드는 로컬 우선 MAS는 이 세 가지를 조합한 구조다.

## 5. 실습 데이터 확인

이번 회사 공유 데모는 `disease.db`와 `disease_info.csv`를 로컬 신뢰 데이터로 사용한다.

운영 환경에서는 이 부분이 MySQL, PostgreSQL, 사내 API, 문서 검색 시스템, 벡터 DB로 대체된다.

In [ ]:
# SQLite 데이터 확인
with sqlite3.connect(DB_PATH) as conn:
    rows = conn.execute("SELECT name, diet, exercise FROM disease").fetchall()

for row in rows:
    print(row)

print("\nCSV 데이터 확인")
with CSV_PATH.open(encoding="utf-8") as f:
    for row in csv.DictReader(f):
        print(row)

## 5-1. RAG는 왜 여기서 필요한가

뒤에서 만드는 MAS는 로컬 DB와 CSV를 먼저 조회한다. 그런데 실제 업무에서는 항상 정확한 key로 질문이 들어오지 않는다.

예를 들어 DB에는 `고혈압`이라는 key가 있는데, 사용자는 이렇게 물을 수 있다.

```text
혈압 높은 사람은 식단을 어떻게 해야 해?
```

이 경우 단순 key 매칭만 하면 `고혈압`을 못 찾을 수 있다. 여기서 RAG가 필요해진다.

```mermaid
flowchart TD
    Q[사용자 질문] --> K{정확한 로컬 key가 있는가?}
    K -- 예 --> Local[DB / CSV 조회]
    K -- 아니오 --> Web[Web fallback]
    Web --> RAG[RAG Agent
내부 문서 검색]
    Local --> Reporter[Reporter]
    RAG --> Reporter
```

이 강의에서 RAG Agent는 간단한 키워드 검색으로 구현하지만, 운영에서는 보통 아래 구조로 확장한다.

```mermaid
flowchart LR
    Docs[사내 문서] --> Chunk[Chunking]
    Chunk --> Embed[Embedding]
    Embed --> VDB[Vector DB]
    User[사용자 질문] --> QEmbed[Query Embedding]
    QEmbed --> VDB
    VDB --> TopK[관련 문서 Top-K]
    TopK --> LLM[LLM 답변 생성]
```

정리하면:

- DB/CSV는 정확한 key가 있을 때 강하다.
- RAG는 표현이 다르거나 문서 기반 근거가 필요할 때 강하다.
- 회사 시스템에서는 **정확한 구조화 데이터는 DB/CSV**, **애매한 자연어 질문은 RAG**로 보완하는 설계가 안정적이다.


## 6. 로컬 우선 MAS 설계

이번 그래프의 의도는 다음과 같다.

```mermaid
flowchart TD
    START([START]) --> Entry[entry_node]
    Entry --> Route{로컬 DB/CSV에 정확한 질병명이 있는가?}
    Route -->|예| DBAgent[db_agent]
    Route -->|예| CSVAgent[csv_agent]
    Route -->|아니오| WebAgent[web_agent]
    DBAgent --> Reporter[reporter]
    CSVAgent --> Reporter
    WebAgent --> NeedRAG{정보 충분?}
    NeedRAG -->|아니오| RAGAgent[rag_agent]
    NeedRAG -->|예| Reporter
    RAGAgent --> Reporter
    Reporter --> END([END])
```

공유 포인트:

- `entry_node`는 답변을 만들지 않고, **어느 경로로 보낼지 판단할 state**를 만든다.
- 로컬에 정확한 키가 있으면 DB/CSV Agent가 병렬로 돈다.
- 로컬에 없으면 Web Agent를 먼저 거치고, 충분하지 않으면 RAG Agent로 넘어간다.
- Reporter는 지금까지 수집된 evidence를 합쳐 최종 답변을 만든다.

In [ ]:
class AgentState(TypedDict, total=False):
    # messages는 사용자 질문, 노드 결과, 최종 답변이 누적되는 대화 공간이다.
    messages: Annotated[Sequence[BaseMessage], add_messages]

    # question은 공유/테스트에서 보기 쉽게 따로 빼둔 사용자 질문이다.
    question: str

    # disease는 로컬 DB/CSV에서 정확히 찾은 질병명이다. 못 찾으면 "미확인"으로 둔다.
    disease: str

    # found=True면 로컬 DB/CSV 경로로, False면 Web/RAG fallback 경로로 보낸다.
    found: bool

    # web_sufficient=True면 웹 결과만으로 reporter로 가고, False면 RAG Agent로 보강한다.
    web_sufficient: bool

    # 여러 노드가 evidence에 동시에 추가하므로 operator.add로 리스트를 이어 붙인다.
    evidence: Annotated[list[str], operator.add]

    # 어떤 agent가 실행됐는지 공유/검증용으로 남긴다.
    used_agents: Annotated[list[str], operator.add]

    # LangGraph 실행 흐름을 테스트하기 위해 노드 방문 기록을 남긴다.
    trace: Annotated[list[str], operator.add]

### State를 시각적으로 보면

앞에서 본 타입을 실제 `AgentState` 기준으로 다시 보면 다음과 같다.

```mermaid
classDiagram
    class AgentState {
        messages: Annotated[Sequence[BaseMessage], add_messages]
        question: str
        disease: str
        found: bool
        web_sufficient: bool
        evidence: Annotated[list[str], operator.add]
        used_agents: Annotated[list[str], operator.add]
        trace: Annotated[list[str], operator.add]
    }

    AgentState --> add_messages : messages reducer
    AgentState --> operator_add : evidence / used_agents / trace reducer
```

- `messages`: 사용자 질문, LLM 응답, Tool 결과가 순서대로 쌓이는 대화 로그다.
- `evidence`: DB/CSV/Web/RAG Agent가 수집한 근거가 누적된다.
- `used_agents`: 어떤 Agent가 실행됐는지 누적된다.
- `trace`: 그래프가 어떤 노드를 지나갔는지 테스트용으로 누적된다.
- `question`, `disease`, `found`, `web_sufficient`는 현재 상태를 나타내는 값이라 보통 덮어써도 된다.


### 6-1. 로컬 우선 MAS 설정값

먼저 로컬 DB/CSV에서 직접 찾을 수 있는 질병명과, RAG fallback에서 사용할 내부 문서를 준비한다.

In [ ]:
KNOWN_DISEASES = ["고혈압", "당뇨", "비만"]  # 로컬 DB/CSV에 정확히 들어 있는 key 목록

# RAG fallback에서 검색할 작은 내부 문서 샘플.
# 운영 환경에서는 사내 문서, 매뉴얼, FAQ, 장애 대응 문서 등을 chunk로 쪼개 vector DB에 넣는다.
INTERNAL_DOCS = [
    {
        "title": "고혈압 생활관리 사내 교육자료",
        "content": "혈압이 높은 사람은 저염식, 채소와 생선 위주의 식단, 칼륨이 풍부한 식품을 고려한다. 빠르게 걷기와 수영처럼 지속 가능한 유산소 운동이 좋다.",
    },
    {
        "title": "당뇨 생활관리 사내 교육자료",
        "content": "당뇨 관리는 정제 탄수화물 제한, 통곡물과 잎채소, 식후 가벼운 걷기, 근력운동이 핵심이다.",
    },
    {
        "title": "비만 생활관리 사내 교육자료",
        "content": "비만 관리는 고단백 저칼로리 식단, 가공식품 줄이기, 유산소와 근력 운동 병행이 중요하다.",
    },
]


### 6-2. 공통 helper와 진입 노드

`entry_node`는 사용자 질문에서 로컬 데이터로 바로 찾을 수 있는 key가 있는지 판단한다.

In [ ]:
def last_user_text(state: AgentState) -> str:
    """State에서 사용자 질문을 안정적으로 꺼낸다."""
    if state.get("question"):
        return state["question"]
    messages = state.get("messages", [])
    return messages[-1].content if messages else ""


def entry_node(state: AgentState):
    """로컬 DB/CSV에 정확한 질병명이 있는지 판단하는 라우팅 준비 노드."""
    question = last_user_text(state)
    disease = next((name for name in KNOWN_DISEASES if name in question), "")

    return {
        "question": question,
        "disease": disease or "미확인",
        "found": bool(disease),
        "trace": ["entry_node"],
    }


def route_after_entry(state: AgentState):
    """로컬 데이터가 있으면 DB/CSV 병렬 fan-out, 없으면 Web fallback."""
    if state.get("found"):
        return ["db_agent", "csv_agent"]
    return "web_agent"


### 6-3. 로컬 정보 수집 Agent: DB Agent와 CSV Agent

로컬 key가 정확히 잡히면 두 Agent가 병렬로 실행되어 서로 다른 로컬 출처를 조회한다.

In [ ]:
def db_agent(state: AgentState):
    """SQLite DB에서 식단과 운동 정보를 조회하는 노드."""
    disease = state["disease"]
    with sqlite3.connect(DB_PATH) as conn:
        row = conn.execute(
            "SELECT diet, exercise FROM disease WHERE name = ?",
            (disease,),
        ).fetchone()

    if not row:
        evidence = f"[DB] {disease} 데이터 없음"
    else:
        diet, exercise = row
        evidence = f"[DB] {disease}: 식단={diet}; 운동={exercise}"

    return {
        "evidence": [evidence],
        "used_agents": ["db_agent"],
        "trace": ["db_agent"],
    }


def csv_agent(state: AgentState):
    """CSV에서 증상과 주의사항을 조회하는 노드."""
    disease = state["disease"]
    with CSV_PATH.open(encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    row = next((r for r in rows if r["name"] == disease), None)
    if not row:
        evidence = f"[CSV] {disease} 데이터 없음"
    else:
        evidence = f"[CSV] {disease}: 증상={row['symptom']}; 주의={row['caution']}"

    return {
        "evidence": [evidence],
        "used_agents": ["csv_agent"],
        "trace": ["csv_agent"],
    }


### 6-4. Web fallback과 RAG Agent

로컬 key가 없거나 웹 결과만으로 부족한 경우를 대비해 RAG Agent로 내부 문서를 보강한다.

In [ ]:
def web_agent(state: AgentState):
    """외부 검색 fallback 자리. 회사 데모에서는 개인정보/외부망 이슈를 고려해 RAG로 넘긴다."""
    return {
        "web_sufficient": False,
        "evidence": ["[Web] 회사 공유 데모에서는 외부 검색 대신 RAG Agent로 내부 문서 fallback을 수행합니다."],
        "used_agents": ["web_agent"],
        "trace": ["web_agent"],
    }


def route_after_web(state: AgentState):
    """웹 결과가 충분하면 reporter, 부족하면 RAG Agent로 보강."""
    return "reporter" if state.get("web_sufficient") else "rag_agent"


### 6-4-1. 작은 RAG 검색기와 RAG Agent

실습에서는 단순 키워드 검색으로 보여주고, 운영에서는 embedding 기반 검색으로 바꾼다.

In [ ]:
def simple_retrieve(query: str, k: int = 2):
    """작은 데모용 RAG 검색기. 운영에서는 embedding + vector DB + reranker로 바꾼다."""
    tokens = set(re.findall(r"[가-힣A-Za-z0-9]+", query))
    scored = []

    for doc in INTERNAL_DOCS:
        doc_text = doc["title"] + " " + doc["content"]
        doc_tokens = set(re.findall(r"[가-힣A-Za-z0-9]+", doc_text))
        score = len(tokens & doc_tokens)

        # 사용자가 '혈압'이라고만 말해도 고혈압 문서가 올라오도록 약한 도메인 힌트를 준다.
        if "혈압" in query and "고혈압" in doc["title"]:
            score += 3

        scored.append((score, doc))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored[:k] if score > 0]


def rag_agent(state: AgentState):
    """내부 문서에서 의미 기반으로 근거를 보강하는 RAG Agent 역할의 노드."""
    docs = simple_retrieve(state["question"], k=2)

    if not docs:
        evidence = "[RAG] 관련 내부 문서를 찾지 못했습니다."
    else:
        evidence = "\n".join(
            f"[RAG] {doc['title']}: {doc['content']}" for doc in docs
        )

    return {
        "evidence": [evidence],
        "used_agents": ["rag_agent"],
        "trace": ["rag_agent"],
    }

### 6-5. Reporter Node

여러 Agent가 모은 근거를 하나의 최종 답변으로 합치는 노드다.

In [ ]:
def reporter(state: AgentState):
    """수집된 evidence를 합쳐 최종 리포트를 만든다. 이 셀은 재현 가능한 테스트를 위해 규칙 기반으로 작성한다."""
    evidence = state.get("evidence", [])
    used_agents = ", ".join(state.get("used_agents", []))

    content = (
        "### 통합 건강관리 리포트\n\n"
        f"질문: {state.get('question', '')}\n\n"
        f"사용 에이전트: {used_agents}\n\n"
        "#### 수집 근거\n"
        + "\n".join(f"- {item}" for item in evidence)
        + "\n\n#### 공유 포인트\n"
        "- 정확한 로컬 키가 있으면 DB/CSV Agent가 병렬로 실행된다.\n"
        "- 로컬 키가 없으면 Web fallback을 거친 뒤 RAG Agent가 내부 문서를 검색한다.\n"
        "- Reporter는 여러 출처의 결과를 하나의 답변으로 정리한다."
    )

    return {
        "messages": [AIMessage(content=content)],
        "trace": ["reporter"],
    }

## 7. 그래프 컴파일

`add_conditional_edges()`에서 리스트를 반환하면 fan-out이 가능하다.

- `return ["db_agent", "csv_agent"]` → 두 노드가 같은 단계에서 실행된다.
- 두 노드가 모두 `reporter`로 연결되어 있으므로, LangGraph가 state를 합쳐 reporter에게 넘긴다.

In [ ]:
builder = StateGraph(AgentState)

builder.add_node("entry_node", entry_node)
builder.add_node("db_agent", db_agent)
builder.add_node("csv_agent", csv_agent)
builder.add_node("web_agent", web_agent)
builder.add_node("rag_agent", rag_agent)
builder.add_node("reporter", reporter)

builder.add_edge(START, "entry_node")

builder.add_conditional_edges(
    "entry_node",
    route_after_entry,
    ["db_agent", "csv_agent", "web_agent"],
)

builder.add_edge("db_agent", "reporter")
builder.add_edge("csv_agent", "reporter")

builder.add_conditional_edges(
    "web_agent",
    route_after_web,
    {"rag_agent": "rag_agent", "reporter": "reporter"},
)

builder.add_edge("rag_agent", "reporter")
builder.add_edge("reporter", END)

graph = builder.compile()
print("graph compiled")

## 8. 그래프 시각화와 X-Ray 검증

강의할 때는 `graph` 그림만 보여줘도 흐름이 잡힌다.

- 일반 그래프: 내가 설계한 노드/엣지 중심
- X-Ray 그래프: 내부 subgraph까지 더 자세히 확인

In [ ]:
def show_graph(compiled_graph, *, xray: bool = False):
    """PNG 렌더링이 되면 이미지로 보여주고, 안 되면 Mermaid 코드로 fallback."""
    try:
        png = compiled_graph.get_graph(xray=xray).draw_mermaid_png()
        display(Image(png))
    except Exception as exc:
        print("PNG 렌더링 실패:", exc)
        print(compiled_graph.get_graph(xray=xray).draw_mermaid())

show_graph(graph)

In [ ]:
# 내부 구조까지 보고 싶을 때 사용
show_graph(graph, xray=True)

## 9. 실행 예시 1: 로컬 데이터가 있는 경우

질문에 `고혈압`, `당뇨`, `비만`처럼 DB/CSV에 있는 정확한 키가 들어가면 `db_agent`와 `csv_agent`가 병렬 실행된다.

In [ ]:
result_local = graph.invoke({
    "messages": [HumanMessage(content="고혈압 환자에게 맞는 식단과 운동을 알려줘")]
})

print("trace:", result_local["trace"])
print("used_agents:", result_local["used_agents"])
display(Markdown(result_local["messages"][-1].content))

## 10. 실행 예시 2: 로컬 키가 정확히 없어서 RAG로 보강하는 경우

사용자가 `고혈압`이라는 정확한 키 대신 `혈압 높은 사람`처럼 표현하면 DB key 조회가 실패할 수 있다.

이럴 때는 Web fallback을 거친 뒤, 내부 문서 기반 RAG Agent가 의미상 가까운 자료를 찾아 보강한다.

In [ ]:
result_rag = graph.invoke({
    "messages": [HumanMessage(content="혈압 높은 사람 생활관리 방법을 알려줘")]
})

print("trace:", result_rag["trace"])
print("used_agents:", result_rag["used_agents"])
display(Markdown(result_rag["messages"][-1].content))

## 11. LLM으로 최종 답변 다듬기

앞의 reporter는 테스트 재현성을 위해 규칙 기반으로 만들었다.

실제 서비스에서는 마지막 reporter에서 LLM을 호출해 문장 품질을 높일 수 있다. 이때도 API 키는 환경변수 `OPENAI_API_KEY`로 읽는다.

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = f"""
아래 수집 근거를 바탕으로 회사 공유용으로 간결하고 신뢰성 있게 건강관리 리포트를 작성해줘.
추측은 하지 말고, 근거에 있는 내용만 사용해.

[근거]
{result_rag['messages'][-1].content}
"""

llm_response = llm.invoke(prompt)
display(Markdown(llm_response.content))

## 12. 테스트: 그래프가 의도대로 흘렀는지 검증

회사 공유에서 중요한 포인트는 “답변이 그럴듯하다”가 아니라, **설계한 경로가 실제로 실행됐는지 검증 가능해야 한다**는 것이다.

아래 테스트는 LLM 출력 내용이 아니라 trace와 agent 경로를 검증한다.

In [ ]:
# 1) 정확한 로컬 키가 있으면 DB/CSV 경로를 탄다.
assert "db_agent" in result_local["used_agents"]
assert "csv_agent" in result_local["used_agents"]
assert "rag_agent" not in result_local["used_agents"]

# 2) 정확한 로컬 키가 없으면 Web fallback 후 RAG 경로를 탄다.
assert "web_agent" in result_rag["used_agents"]
assert "rag_agent" in result_rag["used_agents"]

# 3) 모든 흐름은 reporter를 거쳐 종료된다.
assert result_local["trace"][-1] == "reporter"
assert result_rag["trace"][-1] == "reporter"

print("그래프 경로 검증 통과")

## 13. 고도화 1: Hybrid Search와 RRF

실무 RAG에서는 검색기를 하나만 쓰지 않는 경우가 많다.

예:

- BM25: 키워드가 정확히 맞을 때 강함
- Embedding Search: 의미가 비슷할 때 강함
- Rule/Metadata Search: 업무 규칙이나 권한 필터에 강함

여러 검색 결과를 합칠 때 쓰는 대표 기법이 `RRF(Reciprocal Rank Fusion)`이다.

```mermaid
flowchart LR
    Q[Query] --> B[BM25]
    Q --> E[Embedding]
    Q --> R[Rule Search]
    B --> F[RRF Merge]
    E --> F
    R --> F
    F --> TopK[Final Top-K]
```

`bigzami2/AI_Agent/tests/catalog_rag/test_hybrid_rrf.py`에서는 공통으로 상위에 나온 결과가 최종 상위로 올라오는지 테스트한다.

In [ ]:
def rrf_merge(rank_lists, *, rrf_k: int = 60, top_n: int | None = None):
    """여러 ranking list를 RRF 점수로 합친다."""
    scores = {}

    for rank_list in rank_lists:
        seen_in_this_list = set()
        for rank, item in enumerate(rank_list, start=1):
            doc_id = item[0] if isinstance(item, tuple) else item
            if doc_id in seen_in_this_list:
                continue
            seen_in_this_list.add(doc_id)
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (rrf_k + rank)

    merged = sorted(scores, key=scores.get, reverse=True)
    return merged[:top_n] if top_n else merged

bm25 = ["고혈압_식단", "당뇨_식단", "비만_운동"]
embedding = ["혈압_생활관리", "고혈압_식단", "저염식_가이드"]
rule = ["고혈압_식단", "저염식_가이드"]

merged = rrf_merge([bm25, embedding, rule], top_n=3)
print(merged)
assert merged[0] == "고혈압_식단"

## 14. 고도화 2: Tool Observability

Agent는 도구를 여러 번 호출할 수 있다. 그래서 운영에서는 다음을 기록해야 한다.

- 어떤 도구가 호출됐는가?
- 몇 ms 걸렸는가?
- 성공했는가, 실패했는가?
- 같은 실행 안에서 호출 ID를 추적할 수 있는가?

`bigzami2/AI_Agent/tests/scenarios/test_tool_observability.py`의 핵심은 도구 본문을 뜯어고치지 않고 wrapper로 관측성을 붙이는 것이다.

In [ ]:
def observe_tool(name: str):
    """도구 함수에 호출 시간과 성공/실패 로그를 붙이는 간단한 wrapper."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            started = time.perf_counter()
            status = "ok"
            try:
                return func(*args, **kwargs)
            except Exception as exc:
                status = f"error:{type(exc).__name__}"
                raise
            finally:
                duration_ms = (time.perf_counter() - started) * 1000
                print(f"tool_call name={name} status={status} duration_ms={duration_ms:.1f}")
        return wrapper
    return decorator

@observe_tool("demo_db_lookup")
def demo_db_lookup(name: str):
    return f"{name} 조회 완료"

print(demo_db_lookup("고혈압"))

## 15. 고도화 3: Golden Test와 LLM-as-Judge

Agent 서비스는 매번 다른 답변을 만들 수 있으므로, 일반적인 단위 테스트만으로 충분하지 않다.

운영에서는 보통 세 층으로 평가한다.

```mermaid
flowchart TD
    Scenario[Golden Scenario] --> Run[Agent 실행]
    Run --> Response[응답 평가]
    Run --> Trajectory[경로/노드 평가]
    Run --> Assertion[규칙 평가]
    Response --> Result[RunnerResult]
    Trajectory --> Result
    Assertion --> Result
```

평가 기준 예:

- Response: 최종 답변에 필요한 항목이 들어갔는가?
- Trajectory: `entry_node -> db_agent/csv_agent -> reporter`처럼 의도한 경로를 탔는가?
- Assertion: 금지된 도구를 쓰지 않았는가, 필수 근거가 있는가?
- LLM-as-Judge: 사람이 매번 평가하기 어려운 자연어 품질을 LLM이 보조 평가한다.
- Swarm Judge: 여러 persona judge가 독립 평가하고 합의된 것만 자동 반영한다.

In [ ]:
def evaluate_trajectory(actual_trace: list[str], expected_nodes: list[str]):
    """expected_nodes가 actual_trace 안에 순서대로 등장하는지 확인한다."""
    cursor = 0
    for node in actual_trace:
        if cursor < len(expected_nodes) and node == expected_nodes[cursor]:
            cursor += 1
    missing = expected_nodes[cursor:]
    return {
        "ok": not missing,
        "matched": expected_nodes[:cursor],
        "missing": missing,
    }

print(evaluate_trajectory(result_local["trace"], ["entry_node", "db_agent", "reporter"]))
print(evaluate_trajectory(result_rag["trace"], ["entry_node", "web_agent", "rag_agent", "reporter"]))

## 16. 고도화 4: Provider Policy

회사 운영에서는 모델 이름을 코드 곳곳에 직접 박아두면 관리가 어려워진다.

`bigzami2/AI_Agent/data/seeds/llm_policy.json`처럼 다음을 정책으로 분리할 수 있다.

- 환경별 사용 가능한 LLM 서버
- 역할별 사용 가능한 모델
- vision 지원 여부
- 기본 모델
- 최대 토큰, temperature 기본값

```mermaid
flowchart LR
    Role[사용자 Role] --> Policy[LLM Policy]
    Env[실행 환경] --> Policy
    Purpose[사용 목적] --> Policy
    Policy --> Model[선택 모델]
```

이렇게 하면 개발/운영/로컬 환경에서 같은 Agent 코드를 유지하면서 모델 선택만 정책으로 바꿀 수 있다.

In [ ]:
LLM_POLICY_DEMO = {
    "ROLE_USER": ["openai/gpt-4o-mini"],
    "ROLE_ADMIN": ["openai/gpt-4o", "openai/gpt-4o-mini"],
    "default": ["openai/gpt-4o-mini"],
}

MODEL_CAPABILITIES = {
    "openai/gpt-4o": {"vision": True, "cost": "high", "use": "judge/report"},
    "openai/gpt-4o-mini": {"vision": True, "cost": "low", "use": "routing/draft"},
}


def choose_model(role: str, purpose: str = "draft"):
    candidates = LLM_POLICY_DEMO.get(role, LLM_POLICY_DEMO["default"])
    if purpose == "judge" and "openai/gpt-4o" in candidates:
        return "openai/gpt-4o"
    return candidates[0]

print("ROLE_USER draft:", choose_model("ROLE_USER"))
print("ROLE_ADMIN judge:", choose_model("ROLE_ADMIN", purpose="judge"))

## 17. 문서 처리까지 확장: OCR / VLM / Parser

RAG Agent를 제대로 쓰려면 먼저 문서를 AI가 읽을 수 있는 형태로 만들어야 한다.

```mermaid
flowchart LR
    PDF[PDF / HWP / 이미지] --> Extract[PyMuPDF / OCR / VLM]
    Extract --> Normalize[오타 / 줄바꿈 / 표 구조 보정]
    Normalize --> Parse[JSON / Markdown / HTML Parser]
    Parse --> Chunk[Chunking]
    Chunk --> Embed[Embedding]
    Embed --> VectorDB[Vector DB]
    VectorDB --> RAG[RAG Agent]
```

정리:

- 텍스트 PDF는 `PyMuPDF(import fitz)`로 텍스트 추출이 잘 된다.
- 스캔 PDF나 이미지 문서는 OCR 또는 VLM이 필요하다.
- OCR은 오타, 띄어쓰기, 줄 인식, 문장부호, 표 구조가 깨질 수 있다.
- JSON/Markdown/HTML parser를 이용하면 구조를 최대한 보존하면서 RAG에 넣기 좋다.

## 17-1. 어디를 진짜 Agent로 고도화할까?

현재 실습 그래프는 설명과 테스트가 쉬우도록 `db_agent`, `csv_agent`, `rag_agent`를 일반 노드로 구현했다.

운영에서는 아래 위치를 `create_react_agent()` 기반 Agent로 바꿀 수 있다.

```mermaid
flowchart TD
    Entry[entry_node] --> Router{Routing}
    Router --> DBAgent[DB Agent
SQL Tool 보유]
    Router --> CSVAgent[CSV Agent
File Tool 보유]
    Router --> WebAgent[Web Agent
Tavily Tool 보유]
    WebAgent --> RAGAgent[RAG Agent
Retriever Tool 보유]
    DBAgent --> Reporter
    CSVAgent --> Reporter
    RAGAgent --> Reporter
```

구분:

- 노드 함수: 실행 순서가 명확하고 테스트가 쉬움
- Agent 노드: 도구 선택이나 재시도 판단을 LLM에게 맡길 수 있음
- 회사 서비스에서는 둘을 섞는다. **흐름은 Graph로 통제하고, 판단은 Agent에게 위임**하는 방식이 안정적이다.

In [ ]:
# 운영형으로 바꿀 때의 뼈대 예시다. 이 셀은 설명용이며 바로 실행하지 않아도 된다.
# from langchain_core.tools import tool
# from langgraph.prebuilt import create_react_agent
#
# @tool
# def internal_doc_search(query: str):
#     """사내 문서/RAG 인덱스에서 관련 문서를 검색할 때 사용한다."""
#     docs = retriever.invoke(query)
#     return "\n\n".join(doc.page_content for doc in docs)
#
# rag_agent_runtime = create_react_agent(
#     llm,
#     tools=[internal_doc_search],
#     prompt="너는 사내 문서를 검색해서 근거를 제공하는 RAG 담당 에이전트다.",
# )
#
# builder.add_node("rag_agent", rag_agent_runtime)

## 18. 마지막 응용 예시: 기상청 정보 기반 Human-in-the-loop Agent

이 예시는 회사 업무에서 자주 필요한 **자동화 + 사람 승인** 구조를 보여준다.

목표는 단순히 날씨 문장을 만드는 것이 아니라, 다음 흐름을 LangGraph로 표현하는 것이다.

1. 사용자가 도시를 입력한다.
2. 기상청 RSS에서 날씨 정보를 가져온다.
3. LLM Agent가 사내 공유용 날씨 브리핑 초안을 만든다.
4. 사람이 초안을 보고 `승인`하거나 수정 요청을 남긴다.
5. 승인되면 최종 답변으로 종료하고, 수정 요청이면 LLM이 다시 다듬는다.

핵심 포인트:

- `interrupt()`는 그래프 실행을 일부러 멈추고 사람 입력을 기다리는 기능이다.
- `MemorySaver()`는 멈춘 위치와 State를 저장해서 같은 `thread_id`로 이어 실행할 수 있게 한다.
- `Command(resume=...)`는 사람이 입력한 승인/수정 내용을 그래프에 다시 넣어 재개하는 방법이다.
- 실제 회사 시스템에서는 여기의 `MemorySaver`를 `SqliteSaver`, `PostgresSaver` 같은 영속 저장소로 바꾸는 것이 운영에 더 적합하다.


### 18-0. Human-in-the-loop 예시 준비: import와 기상청 설정

먼저 필요한 모듈과 도시별 기상청 RSS 코드, fallback 데이터를 준비한다.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
import requests
import xml.etree.ElementTree as ET

# 기상청 RSS 예시에서 사용할 지역 코드.
# 실제 서비스에서는 행정구역 코드 테이블을 따로 관리하거나, 사용자 위치를 좌표로 변환해 사용한다.
KMA_ZONE_CODES = {
    "서울": "1159068000",
    "부산": "2644058000",
    "대구": "2726069000",
    "광주": "2917063000",
    "대전": "3017063000",
}

# 기상청 호출이 실패해도 강의가 끊기지 않도록 준비한 fallback 데이터.
# 회사 발표에서는 네트워크, 방화벽, 기관 API 변경 때문에 외부 호출이 실패할 수 있다.
KMA_FALLBACK = {
    "서울": "서울: 흐림, 기온 24도, 강수 가능성 낮음. 오후 외근 시 우산은 선택 사항.",
    "부산": "부산: 맑음, 기온 26도, 해안가 바람 약간. 야외 일정 가능.",
    "대구": "대구: 구름 많음, 기온 28도. 장시간 야외 활동은 수분 보충 필요.",
}


### 18-0-1. Human-in-the-loop State 정의

이 State는 날씨 원천 정보, LLM 초안, 사람 피드백, 최종 결과를 순서대로 담는다.

In [ ]:
class WeatherReviewState(TypedDict, total=False):
    city: str             # 사용자가 요청한 도시명
    weather_raw: str      # 기상청 또는 fallback에서 가져온 원천 날씨 정보
    draft: str            # LLM이 작성한 사내 공유용 초안
    human_feedback: str   # 사람이 승인하거나 수정 요청한 내용
    final: str            # 최종 확정된 답변


### 18-0-2. 기상청 정보 수집 노드

외부 API 호출은 실패할 수 있으므로 timeout과 fallback을 같이 둔다.

In [ ]:
def fetch_kma_weather(state: WeatherReviewState):
    """도시명으로 기상청 RSS를 조회하고, 실패하면 강의용 fallback 데이터를 반환한다."""
    city = state.get("city", "서울")
    zone = KMA_ZONE_CODES.get(city, KMA_ZONE_CODES["서울"])
    url = f"https://www.weather.go.kr/w/rss/dfs/hr1-forecast.do?zone={zone}"

    try:
        response = requests.get(url, timeout=5)  # 외부 API는 timeout을 꼭 둔다.
        response.raise_for_status()
        root = ET.fromstring(response.content)

        # RSS 안의 description 텍스트를 가능한 범위에서 추출한다.
        descriptions = [el.text for el in root.iter() if el.tag.endswith("description") and el.text]
        weather_text = descriptions[-1] if descriptions else "기상청 RSS에서 상세 설명을 찾지 못했습니다."
        return {"weather_raw": f"{city}: {weather_text}"}

    except Exception as e:
        # 실습 중 외부 호출 실패가 핵심 흐름을 망치지 않도록 fallback한다.
        fallback = KMA_FALLBACK.get(city, KMA_FALLBACK["서울"])
        return {"weather_raw": f"[기상청 호출 실패로 fallback 사용] {fallback} / 원인: {type(e).__name__}"}


### 18-0-3. LLM 초안 생성 노드

기상청 원천 정보를 사람이 읽기 쉬운 사내 브리핑 초안으로 바꾼다.

In [ ]:
def draft_weather_briefing(state: WeatherReviewState):
    """원천 날씨 정보를 사내 공유용 브리핑 초안으로 바꾼다."""
    prompt = f"""
너는 사내 운영 공유용 날씨 브리핑 담당 AI Agent다.
아래 원천 날씨 정보를 보고 직원이 바로 이해할 수 있게 3줄로 정리하라.

조건:
- 1줄: 현재 날씨 요약
- 1줄: 출근/외근 관점의 주의사항
- 1줄: 필요한 준비물

[원천 정보]
{state['weather_raw']}
"""
    response = llm.invoke(prompt)  # 앞에서 만든 ChatOpenAI 인스턴스를 사용한다.
    return {"draft": response.content}


### 18-0-4. 사람 검토 지점과 라우팅 함수

`interrupt()`가 실행되면 그래프는 여기서 멈추고, 사람 입력을 기다린다.

In [ ]:
def human_review(state: WeatherReviewState):
    """초안을 사람에게 보여주고 승인 또는 수정 요청을 받는 중단 지점."""
    feedback = interrupt({
        "질문": "아래 날씨 브리핑 초안을 승인할까요? 수정이 필요하면 수정 요청을 적어주세요.",
        "초안": state["draft"],
        "입력예시": "승인 / 더 짧게 줄여줘 / 외근자 주의사항을 강조해줘",
    })
    return {"human_feedback": feedback}


def route_after_review(state: WeatherReviewState):
    """사람 입력이 승인인지 수정 요청인지 판단해 다음 노드를 고른다."""
    feedback = str(state.get("human_feedback", ""))
    if "승인" in feedback or "좋아" in feedback or "확정" in feedback:
        return "approve"
    return "revise"


### 18-0-5. 승인/수정 후 처리 노드

사람이 승인하면 그대로 확정하고, 수정 요청이면 LLM에게 다시 다듬게 한다.

In [ ]:
def revise_with_feedback(state: WeatherReviewState):
    """사람의 수정 요청을 반영해 초안을 다시 작성한다."""
    prompt = f"""
아래 날씨 브리핑 초안을 사람의 수정 요청에 맞게 다시 작성하라.

[기존 초안]
{state['draft']}

[사람 수정 요청]
{state['human_feedback']}
"""
    response = llm.invoke(prompt)
    return {"final": response.content}


def finalize_weather_briefing(state: WeatherReviewState):
    """사람이 승인한 초안을 최종 답변으로 확정한다."""
    return {"final": state["draft"]}


### 18-0-6. Human-in-the-loop 그래프 조립

`interrupt()`를 사용하는 그래프는 반드시 checkpointer를 붙여야 한다.

In [ ]:
weather_builder = StateGraph(WeatherReviewState)
weather_builder.add_node("fetch_kma_weather", fetch_kma_weather)       # 1단계: 외부 정보 수집
weather_builder.add_node("draft_agent", draft_weather_briefing)        # 2단계: LLM 초안 생성
weather_builder.add_node("human_review", human_review)                 # 3단계: 사람 승인 지점
weather_builder.add_node("revise_agent", revise_with_feedback)         # 4-A: 수정 요청 반영
weather_builder.add_node("finalize", finalize_weather_briefing)        # 4-B: 승인 시 확정

weather_builder.add_edge(START, "fetch_kma_weather")
weather_builder.add_edge("fetch_kma_weather", "draft_agent")
weather_builder.add_edge("draft_agent", "human_review")
weather_builder.add_conditional_edges(
    "human_review",
    route_after_review,
    {
        "approve": "finalize",      # 사람이 승인하면 바로 최종 확정
        "revise": "revise_agent",   # 수정 요청이면 LLM이 다시 다듬음
    },
)
weather_builder.add_edge("revise_agent", END)
weather_builder.add_edge("finalize", END)

# interrupt를 쓰려면 checkpointer가 필요하다.
# MemorySaver는 실습용이며, 런타임이 꺼지면 저장된 중단 상태가 사라진다.
weather_graph = weather_builder.compile(checkpointer=MemorySaver())


### 18-1. 그래프 구조 확인

일반 그래프는 상위 흐름만 보여주고, X-Ray 그래프는 내부 노드까지 더 자세히 보여준다.

휴먼인더루프 예시에서는 `draft_agent -> human_review` 지점에서 그래프가 멈춘 뒤, 사람이 `Command(resume=...)`로 재개한다.


In [ ]:
show_graph(weather_graph)
show_graph(weather_graph, xray=True)


### 18-2. 실행 1: 사람 승인 대기 지점까지 실행

아래 셀을 실행하면 그래프가 `human_review`에서 멈춘다.

출력의 `__interrupt__` 안에 사람이 검토해야 할 초안이 들어 있다.


In [ ]:
weather_config = {"configurable": {"thread_id": "weather-demo-seoul"}}

first_result = weather_graph.invoke(
    {"city": "서울"},      # 사용자가 요청한 도시
    config=weather_config,  # 같은 thread_id를 써야 중단 지점에서 이어갈 수 있다.
)

first_result["__interrupt__"][0].value


### 18-3. 실행 2: 사람이 `input()`으로 승인하거나 수정 요청하기

이 셀은 Human-in-the-loop의 핵심을 가장 직접적으로 보여준다.

1. 위 셀에서 그래프가 `human_review`에서 멈춘다.
2. 사람이 초안을 보고 `승인` 또는 수정 요청을 입력한다.
3. `Command(resume=...)`가 그 입력을 그래프에 다시 넣고, 같은 `thread_id`에서 이어 실행한다.

입력 예시:

- `승인`
- `외근자 주의사항을 더 강조해줘`
- `3줄보다 더 짧게 요약해줘`


In [ ]:
# 실제 강의에서는 이 셀을 실행한 뒤 사람이 직접 승인/수정 요청을 입력한다.
# 입력값은 interrupt로 멈춰 있던 human_review 노드의 결과로 들어간다.
human_input = input("승인 또는 수정 요청을 입력하세요: ")

human_result = weather_graph.invoke(
    Command(resume=human_input),  # 사람이 입력한 값을 그래프에 다시 주입한다.
    config=weather_config,        # 실행 1과 같은 thread_id라야 중단 지점에서 이어진다.
)

print(human_result["final"])


In [ ]:
# 자동 시연용 예시: 수정 요청 경로를 빠르게 보여주고 싶을 때만 실행한다.
# 위 input 셀과 같은 thread_id를 이미 끝까지 실행했다면, 새 thread_id를 써야 한다.
weather_config_revise = {"configurable": {"thread_id": "weather-demo-seoul-revise"}}

_ = weather_graph.invoke({"city": "서울"}, config=weather_config_revise)
revised_result = weather_graph.invoke(
    Command(resume="외근자 주의사항을 더 강조하고, 준비물을 짧게 써줘"),
    config=weather_config_revise,
)

print(revised_result["final"])


### 18-4. 더 잘 만들기 위한 갈림길

여기까지의 실습은 기초 역량을 기르기에 충분한 흐름이다. 실제 회사 PoC나 운영 시스템으로 고도화할 때는 아래 갈림길에서 하나씩 붙이면 된다.

| 갈림길 | 붙이는 기능 | 왜 필요한가 | 먼저 해볼 일 |
|---|---|---|---|
| 검색 품질 | **BM25 + Embedding + Rule Search** | 키워드 일치, 의미 유사도, 업무 규칙/권한 필터를 함께 보기 위해 | 기존 RAG 검색기를 3개 검색 결과로 나누기 |
| 검색 결과 병합 | **RRF Merge** | 여러 검색기가 준 순위를 안정적으로 합치기 위해 | BM25/Embedding/Rule 결과를 RRF 점수로 합쳐 Top-K 만들기 |
| 도구 관측 | **Tool Observability** | Agent가 어떤 도구를 언제, 몇 번, 어떤 입력으로 호출했는지 보기 위해 | tool wrapper로 call_id, latency, success/error 기록하기 |
| 회귀 테스트 | **Golden Test** | 프롬프트나 모델을 바꿔도 기존 핵심 시나리오가 깨지지 않는지 확인하기 위해 | 대표 질문 10~20개와 기대 조건을 golden scenario로 저장하기 |
| 자연어 평가 | **LLM-as-Judge** | 사람이 매번 읽기 어려운 답변 품질을 자동 보조 평가하기 위해 | 답변의 근거성, 완전성, 금지어, 형식 준수 여부를 Judge prompt로 평가하기 |
| 모델 운영 정책 | **Provider Policy** | 모델명, temperature, max token, vision 지원 여부를 코드 밖에서 관리하기 위해 | 역할별 모델 정책 JSON/YAML을 만들고 코드에서는 policy를 읽기 |

강의에서 이미 연결한 위치는 다음과 같다.

- `BM25 + Embedding + Rule Search`와 `RRF Merge`는 **13. Hybrid Search와 RRF**에서 다룬다.
- `Tool Observability`는 **14. Tool Observability**에서 wrapper 방식으로 다룬다.
- `Golden Test`와 `LLM-as-Judge`는 **15. Golden Test와 LLM-as-Judge**에서 평가 구조로 다룬다.
- `Provider Policy`는 **16. Provider Policy**에서 모델 정책 분리로 다룬다.
- 마지막 기상청 Human-in-the-loop 예시는 사람이 승인하거나 수정 요청을 입력하는 운영 통제 지점의 예시다.

즉, 이 자료는 `기초 문법`에서 끝나는 자료가 아니라 **검색 → 관측 → 평가 → 모델 정책 → 사람 승인**까지 확장 가능한 뼈대를 보여주는 자료다.


### 18-5. 강의 마무리: 오늘 가져갈 설계 기준

이 노트북의 최종 메시지는 하나다.

**AI Agent를 잘 만든다는 것은 LLM에게 모든 것을 맡기는 것이 아니라, 사람이 통제할 흐름과 LLM이 판단할 영역을 분리하는 것이다.**

오늘 실습에서 확인한 기준:

1. **정해진 업무 흐름은 LangGraph로 고정한다.**
   - `START`, `Node`, `Edge`, `END`로 실행 순서를 명확히 만든다.
   - 조건이 필요한 곳은 `add_conditional_edges()`로 분기한다.

2. **입력마다 달라지는 판단은 Agent나 LLM에게 맡긴다.**
   - 도구 선택, 검색 보강, 초안 작성, 요약처럼 매번 판단이 필요한 부분에 LLM을 둔다.
   - 단, 중요한 판단은 테스트와 검증 기준을 함께 둔다.

3. **로컬 데이터가 있으면 먼저 로컬을 신뢰한다.**
   - DB, CSV, 사내 문서처럼 출처가 명확한 데이터가 우선이다.
   - 부족한 경우에만 Web, RAG, LLM 추론으로 보강한다.

4. **운영 시스템에는 사람 승인 지점을 넣을 수 있다.**
   - `interrupt()`로 그래프를 멈추고 사람이 검토한다.
   - `Command(resume=...)`로 같은 흐름을 이어간다.
   - 실습은 `MemorySaver`, 운영은 DB 기반 checkpointer를 고려한다.

5. **실무형 AI Agent는 관측과 평가까지 포함한다.**
   - 그래프 시각화, X-Ray, trace, tool log, golden test, LLM-as-Judge를 통해 동작을 확인한다.

결국 회사 업무용 LangGraph Agent의 기본형은 다음 구조로 정리할 수 있다.

```mermaid
flowchart LR
    A[사용자 요청] --> B[정해진 업무 흐름
LangGraph]
    B --> C{로컬 데이터 충분?}
    C -- 예 --> D[DB / CSV / 사내 시스템]
    C -- 아니오 --> E[RAG / Web / Tool Agent]
    D --> F[Reporter]
    E --> F
    F --> G{사람 검토 필요?}
    G -- 예 --> H[Human-in-the-loop]
    G -- 아니오 --> I[최종 응답]
    H --> I
```
